# Retraining the neural network models on the train + validation rows

The evaluation plan:

1. **Train-only models** (`temporal_ffn`, `static_temporal_ffn`, `two_stage_ffn_rf`, fitted on the train months 2021-01..2024-03 by the notebooks in `single_stage_models/neural_network` and `two_stage_models/neural_network`) are scored on the held-out validation rows.
2. **This notebook** refits the same three models on the train + validation rows (2021-01..2025-03, 149,999 rows).
3. The test predictions of the **retrained** models are the ones sent to Kaggle.

Nothing here overwrites a train-only model or prediction: everything gets the prefix `retrained_`.

* **Epochs.** With no held-out rows left there is nothing to stop early on, so each network is trained for the epoch its train-only run stopped at (`best_epoch` in the saved model: the epoch of the best validation RMSE), with the same architecture, learning rate, batch size, weight decay and seed.
  The train-only notebooks must have been run with early stopping first (`temporal_ffn.ipynb`, `static_temporal_ffn.ipynb`, `two_stage_nn_forest.ipynb`, in this order). The retrained fit sees 31% more rows per epoch, so it takes 31% more optimiser steps per epoch than the train-only fit did.
* **Inputs.** The scalers stay as the data pipeline fitted them (the temporal and numeric static features on the train months, the ordinal static features on the train rows), so the retrained networks see inputs on the same scale as the train-only ones.
* **Two-stage model.** Stage 1 is the retrained `temporal_ffn` network (the level of each month). Stage 2 is a random forest refitted on all the rows with the settings chosen on the validation rows by `two_stage_nn_forest.ipynb` (read from its saved forest): target = rent / stage-1 level of the month, weighted by `level^2`.
* **Predictions.** Same files and row sets as the train-only models, so the evaluation code reads them unchanged: `train/retrained_<model>.csv` holds the 114,841 train rows, `val/retrained_<model>.csv` the 35,158 validation rows, `test/retrained_<model>.csv` the 50,000 test rows, `levels/retrained_<model>.csv`
  the fixed-flat level of each month. **The retrained models were fitted on both the train and the validation rows, so both of those files are in-sample.** The validation scores of the train-only models are the only held-out accuracy.
* **Outputs.** The models and the training statistics (loss per epoch) in `results/models/` and `results/tables/`, the predictions in `03_evaluation/predictions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

sys.path.insert(0, str(Path.cwd().parent))          # 02_modelling: nn_utils.py is shared with the train-only notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

MODELLING = Path.cwd().parent
TRAIN_ONLY_MODELS = {                               # the models saved by the train-only notebooks
    'temporal_ffn': MODELLING / 'two_stage_models' / 'neural_network' / 'results' / 'models' / 'temporal_ffn.pt',
    'static_temporal_ffn': MODELLING / 'single_stage_models' / 'neural_network' / 'results' / 'models' / 'static_temporal_ffn.pt',
    'two_stage_ffn_rf': MODELLING / 'two_stage_models' / 'neural_network' / 'results' / 'models' / 'two_stage_ffn_rf_forest.joblib'}
PREFIX = 'retrained_'
NAMES = {name: PREFIX + name for name in TRAIN_ONLY_MODELS}
nu.use_results_dir(Path.cwd() / 'results')
for retrained in NAMES.values():
    nu.ensure_output_dirs(retrained)

## 1. Data

The same rows as the train-only notebooks. All 149,999 rows with a rent (`train` and `val`) are trained on; the `sample_type` flags are only used to split the predictions into the `train` and `val` files.

In [2]:
temporal = nu.load_temporal()
TEMPORAL = list(temporal.columns)
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)
fixed = np.isin(month, nu.Q1_2025)                                                   # the fixed flats: the rows of 2025-01..03 (their features only, never their rents)

print(f'{len(TEMPORAL)} temporal + {len(STATIC)} static features; retraining on {len(y):,} rows (months 0..{month.max()}): '
      f'{fit.sum():,} train rows + {val.sum():,} validation rows; test rows {len(test):,}')

177 temporal + 151 static features; retraining on 149,999 rows (months 0..50): 114,841 train rows + 35,158 validation rows; test rows 50,000


## 2. Settings of the train-only models

The architecture, optimiser settings, seed and number of epochs of each train-only network, and the forest settings of the two-stage model, read from the saved train-only models.

In [3]:
def refit_args(config):
    """fit_ffn arguments of a train-only network: its settings, trained for the epoch it stopped at (no validation rows, so no early stopping)."""
    assert config.get('best_epoch'), 'the train-only model was saved without early stopping: rerun its notebook first'
    return dict(epochs=config['best_epoch'], hidden=config['hidden'], activation=config['activation'], batch_size=config['batch_size'],
                learning_rate=config['learning_rate'], weight_decay=config['weight_decay'], seed=config['seed'])

temporal_old, ckpt = nu.load_ffn(TRAIN_ONLY_MODELS['temporal_ffn'])
assert ckpt['temporal_features'] == TEMPORAL, 'temporal_ffn.pt was fitted on other temporal features'
temporal_args = refit_args(temporal_old['config'])

static_old, ckpt = nu.load_ffn(TRAIN_ONLY_MODELS['static_temporal_ffn'])
assert ckpt['temporal_features'] == TEMPORAL and ckpt['static_features'] == STATIC, 'static_temporal_ffn.pt was fitted on other features'
static_args = refit_args(static_old['config'])

forest_old = joblib.load(TRAIN_ONLY_MODELS['two_stage_ffn_rf'])
assert forest_old['static_features'] == STATIC and forest_old['stage1_model'] == 'temporal_ffn.pt', 'two_stage_ffn_rf_forest.joblib was fitted on other features or another stage-1 network: rerun two_stage_nn_forest.ipynb'
RF_PARAMS = forest_old['rf_params']

print('temporal_ffn       ', temporal_args)
print('static_temporal_ffn', static_args)
print('forest             ', RF_PARAMS)

temporal_ffn        {'epochs': 11, 'hidden': [64, 16], 'activation': 'leaky_relu', 'batch_size': 2048, 'learning_rate': 0.001, 'weight_decay': 0.0, 'seed': 0}
static_temporal_ffn {'epochs': 17, 'hidden': [64, 16], 'activation': 'leaky_relu', 'batch_size': 2048, 'learning_rate': 0.001, 'weight_decay': 0.0, 'seed': 0}
forest              {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6, 'max_features': 0.33, 'min_samples_leaf': 10}


## 3. `retrained_temporal_ffn`

The temporal-features network on the rows of months 0..50. Each flat of a month gets that month's level, so the level of the same flats in every month is the level itself.

In [4]:
outputs = {}                                                                          # per retrained model: the predictions of the train / val / test rows, the level by month, the training statistics

temporal_model = nu.fit_ffn(temporal, month, y, **temporal_args)                      # no valid=: every labelled row is trained on
stage1_level = nu.month_levels(temporal_model, temporal)                              # months 0..66
outputs['temporal_ffn'] = {'model': temporal_model, 'level': stage1_level,
                           'pred': {'train': stage1_level.loc[month[fit]].to_numpy(), 'val': stage1_level.loc[month[val]].to_numpy(), 'test': stage1_level.loc[test['month']].to_numpy()}}
print(temporal_model['history'].iloc[[0, -1]].round(1).to_string(index=False))
print(stage1_level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level').T.to_string())

 epoch  train_mse_std  train_rmse_dollar
     1            0.7              634.1
    11            0.7              618.1
       2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level   3102.0   3127.0   3188.0   3150.0   3145.0   3149.0   3171.0   3135.0   3256.0   3268.0   3288.0   3246.0   3295.0   3291.0   3250.0   3231.0   3242.0   3202.0   3329.0   3297.0   3271.0   3383.0   3361.0


## 4. `retrained_static_temporal_ffn`

The static + temporal network on the rows of months 0..50. The fixed flats are the rows of 2025-01..03, predicted as if approved in each month 0..66.

In [5]:
static_model = nu.fit_ffn(temporal, month, y, X=X, **static_args)
pred_all = nu.predict_ffn(static_model, temporal, month, X)
outputs['static_temporal_ffn'] = {'model': static_model, 'level': nu.fixed_flat_levels(static_model, temporal, X[fixed], range(nu.N_MONTHS)),
                                  'pred': {'train': pred_all[fit], 'val': pred_all[val], 'test': nu.predict_ffn(static_model, temporal, test['month'].to_numpy(), X_test)}}
print(static_model['history'].iloc[[0, -1]].round(1).to_string(index=False))
print(outputs['static_temporal_ffn']['level'].loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

 epoch  train_mse_std  train_rmse_dollar
     1            0.6              576.1
    17            0.4              481.9
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3070.0   3090.0   3117.0   3096.0   3092.0   3097.0   3111.0   2990.0   3229.0   3262.0   2930.0   3306.0   3288.0   3283.0   3110.0   3218.0   2994.0   3178.0   3416.0   3376.0   3398.0   3300.0   3135.0


## 5. `retrained_two_stage_ffn_rf`

Stage 1 is the retrained temporal network of section 3. Stage 2: the forest, with the train-only settings, refitted on all the rows: target = rent / (in-sample stage-1 level of its month), weighted by `level^2`.
Prediction = `level of the month x ratio of the flat`; the level of the same flats in each month is level(month) x the mean ratio of the fixed flats.

In [6]:
level_w = stage1_level.loc[month].to_numpy()
forest = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X, y / level_w, sample_weight=level_w ** 2)
ratio_all = forest.predict(X)
pred_all = level_w * ratio_all
outputs['two_stage_ffn_rf'] = {'model': forest, 'level': stage1_level * forest.predict(X[fixed]).mean(),
                               'pred': {'train': pred_all[fit], 'val': pred_all[val], 'test': stage1_level.loc[test['month']].to_numpy() * forest.predict(X_test)}}
print(f'in-sample ratio mean {ratio_all.mean():.3f}')
print(outputs['two_stage_ffn_rf']['level'].loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

in-sample ratio mean 0.999
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3099.0   3124.0   3184.0   3146.0   3141.0   3145.0   3167.0   3131.0   3252.0   3264.0   3284.0   3242.0   3291.0   3287.0   3246.0   3227.0   3238.0   3198.0   3325.0   3293.0   3267.0   3379.0   3357.0


## 6. Save the models, the training statistics and the predictions

The networks and the forest in `results/models/`, the loss per epoch of the two networks in `results/tables/<model>/epoch_losses.csv` (the RMSE of the training rows: no rows are held out), a one-line summary of each fit in `results/tables/retrained_summary.csv`,
and the predictions in `03_evaluation/predictions/{train,val,test,levels}/retrained_<model>.csv`.

In [7]:
NOTE = 'refitted on the train + validation rows (2021-01..2025-03) for the number of epochs of the train-only model'
paths = [nu.MODEL_DIR / f'{NAMES["temporal_ffn"]}.pt', nu.MODEL_DIR / f'{NAMES["static_temporal_ffn"]}.pt', nu.MODEL_DIR / f'{NAMES["two_stage_ffn_rf"]}_forest.joblib']
nu.save_ffn(outputs['temporal_ffn']['model'], paths[0], temporal_features=TEMPORAL, note=NOTE)
nu.save_ffn(outputs['static_temporal_ffn']['model'], paths[1], static_features=STATIC, temporal_features=TEMPORAL, note=NOTE)
joblib.dump({'forest': forest, 'static_features': STATIC, 'rf_params': RF_PARAMS, 'target': 'rent / stage-1 level of the month, weighted by level^2', 'stage1_model': f'{NAMES["temporal_ffn"]}.pt', 'note': NOTE},
            paths[2], compress=3)

summary = []
for name, out in outputs.items():
    retrained = NAMES[name]
    if name != 'two_stage_ffn_rf':
        paths += [nu.save_table(out['model']['history'], retrained, 'epoch_losses.csv', index=False)]
    else:                                                                                         # the stage-1 network's losses are those of retrained_temporal_ffn
        paths += [nu.save_table(pd.Series(RF_PARAMS, name='value').to_frame(), retrained, 'rf_params.csv', index_label='parameter')]
    paths += [nu.save_predictions(retrained, 'train', train[fit], out['pred']['train']),          # the 114,841 train rows (in-sample)
              nu.save_predictions(retrained, 'val', train[val], out['pred']['val']),              # the 35,158 validation rows (in-sample too: they were trained on)
              nu.save_predictions(retrained, 'test', test, out['pred']['test']),                  # the 50,000 test rows (a forecast)
              nu.save_levels(retrained, out['level'])]
    epochs = out['model']['config']['epochs_run'] if name != 'two_stage_ffn_rf' else outputs['temporal_ffn']['model']['config']['epochs_run']
    summary.append({'model': retrained, 'rows_fitted': len(y), 'epochs': epochs, 'in_sample_rmse_train_rows': nu.rmse(y[fit], out['pred']['train']), 'in_sample_rmse_validation_rows': nu.rmse(y[val], out['pred']['val'])})
summary = pd.DataFrame(summary)
summary.to_csv(nu.TABLE_DIR / 'retrained_summary.csv', index=False)
paths += [nu.TABLE_DIR / 'retrained_summary.csv']
print(summary.round(1).to_string(index=False))
print('\nsaved:', *paths, sep='\n  ')

                        model  rows_fitted  epochs  in_sample_rmse_train_rows  in_sample_rmse_validation_rows
       retrained_temporal_ffn       149999      11                      604.7                           658.9
retrained_static_temporal_ffn       149999      17                      481.1                           477.3
   retrained_two_stage_ffn_rf       149999      11                      451.7                           440.3

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/models/retrained_temporal_ffn.pt
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/models/retrained_static_temporal_ffn.pt
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_retraining/results/models/retrained_two_stage_ffn_rf_forest.joblib
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/neural_network_re

## 7. Check against the train-only predictions

The retrained files must cover the same rows in the same order as the train-only ones (the evaluation matches models by `ROW_ID`). The mean test prediction of a retrained model should sit near the train-only one, and its in-sample error on the validation rows should be lower than the train-only held-out error.

In [8]:
rows = []
for name, retrained in NAMES.items():
    for split in nu.SPLITS:
        old = pd.read_csv(nu.PRED_DIR / split / f'{name}.csv', usecols=['ROW_ID', 'prediction'])
        new = pd.read_csv(nu.PRED_DIR / split / f'{retrained}.csv', usecols=['ROW_ID', 'prediction'])
        assert len(old) == len(new) and (old['ROW_ID'].to_numpy() == new['ROW_ID'].to_numpy()).all(), (name, split)
        rows.append({'model': name, 'split': split, 'rows': len(new), 'mean_train_only': old['prediction'].mean(), 'mean_retrained': new['prediction'].mean(),
                     'rmse_between_the_two': nu.rmse(old['prediction'], new['prediction'])})
print(pd.DataFrame(rows).round(1).to_string(index=False))

              model split   rows  mean_train_only  mean_retrained  rmse_between_the_two
       temporal_ffn train 114841           2605.9          2593.0                  31.5
       temporal_ffn   val  35158           3094.7          3136.5                  65.7
       temporal_ffn  test  50000           3221.7          3271.5                  80.1
static_temporal_ffn train 114841           2592.1          2578.7                  72.0
static_temporal_ffn   val  35158           3116.4          3093.6                 114.3
static_temporal_ffn  test  50000           3228.5          3212.4                 160.9
   two_stage_ffn_rf train 114841           2593.1          2589.4                  55.6
   two_stage_ffn_rf   val  35158           3083.2          3134.1                  97.2
   two_stage_ffn_rf  test  50000           3210.9          3271.7                 108.0
